In [23]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score

import pickle


In [24]:
def generate_object_features(n_features, n_samples, rng):
    """Генерирует признаки для одного объекта с 4 типами данных"""
    types = ["binary", "nominal", "ordinal", "quantitative"]
    data = {}
    for i in range(n_features):
        t = types[i % 4]
        col_name = f"feat_{i}_{t}"
        
        if t == "binary":
            data[col_name] = rng.integers(0, 2, size=n_samples)
        elif t == "nominal":
            data[col_name] = rng.choice(["A", "B", "C", "D"], size=n_samples)
        elif t == "ordinal":
            data[col_name] = rng.integers(0, 5, size=n_samples)
        elif t == "quantitative":
            data[col_name] = rng.normal(loc=0.0, scale=1.0, size=n_samples)
            
    return pd.DataFrame(data)

In [25]:
def custom_collision_rule(obj1: pd.DataFrame, obj2: pd.DataFrame) -> pd.Series:
    """
    Вычисляет факт коллизии между двумя объектами на основе их признаков.
    
    Логика правила:
    1. Количественные признаки: если евклидово расстояние по числовым признакам < порога → риск коллизии
    2. Номинальные признаки: если есть совпадение хотя бы одного номинального признака → повышенный риск
    3. Порядковые признаки: если разница по порядковым признакам ≤ 1 → объекты "близки" по состоянию
    4. Бинарные признаки: если оба объекта активны (значение 1) → возможна коллизия
    
    Коллизия = 1, если выполняются ≥ 2 из 4 условий выше.
    """
    n_samples = len(obj1)
    collision_conditions = np.zeros(n_samples, dtype=int)
    
    quant_cols1 = [c for c in obj1.columns if c.endswith('_quantitative')]
    quant_cols2 = [c for c in obj2.columns if c.endswith('_quantitative')]
    
    if quant_cols1 and quant_cols2:
        dist = np.zeros(n_samples)
        for c1, c2 in zip(quant_cols1, quant_cols2):
            std1 = obj1[c1].std() if obj1[c1].std() > 0 else 1
            std2 = obj2[c2].std() if obj2[c2].std() > 0 else 1
            diff = ((obj1[c1] / std1) - (obj2[c2] / std2)) ** 2
            dist += diff
        dist = np.sqrt(dist / len(quant_cols1))
        collision_conditions += (dist < 1.5).astype(int)
    
    nominal_cols1 = [c for c in obj1.columns if c.endswith('_nominal')]
    nominal_cols2 = [c for c in obj2.columns if c.endswith('_nominal')]
    
    if nominal_cols1 and nominal_cols2:
        nominal_match = np.zeros(n_samples, dtype=bool)
        for c1, c2 in zip(nominal_cols1, nominal_cols2):
            nominal_match |= (obj1[c1] == obj2[c2])
        collision_conditions += nominal_match.astype(int)
    
    ordinal_cols1 = [c for c in obj1.columns if c.endswith('_ordinal')]
    ordinal_cols2 = [c for c in obj2.columns if c.endswith('_ordinal')]
    
    if ordinal_cols1 and ordinal_cols2:
        ordinal_close = np.zeros(n_samples, dtype=bool)
        for c1, c2 in zip(ordinal_cols1, ordinal_cols2):
            ordinal_close |= (np.abs(obj1[c1] - obj2[c2]) <= 1)
        collision_conditions += ordinal_close.astype(int)
    
    binary_cols1 = [c for c in obj1.columns if c.endswith('_binary')]
    binary_cols2 = [c for c in obj2.columns if c.endswith('_binary')]
    
    if binary_cols1 and binary_cols2:
        both_active = np.zeros(n_samples, dtype=bool)
        for c1, c2 in zip(binary_cols1, binary_cols2):
            both_active |= ((obj1[c1] == 1) & (obj2[c2] == 1))
        collision_conditions += both_active.astype(int)
    
    return pd.Series(collision_conditions >= 2, index=obj1.index)

In [26]:
sizes = [50, 300, 750, 1500]
n_features_list = [5, 9, 12]

In [27]:
for size in sizes:
    for nf in n_features_list:
        rng = np.random.default_rng(seed=67)
        
        n_obj1 = nf // 2 + (nf % 2)
        n_obj2 = nf // 2
        
        df1 = generate_object_features(n_obj1, size, rng)
        df2 = generate_object_features(n_obj2, size, rng)
        
        df1.columns = [f"obj1_{c}" for c in df1.columns]
        df2.columns = [f"obj2_{c}" for c in df2.columns]
        
        df = pd.concat([df1, df2], axis=1)
        df["collision"] = custom_collision_rule(df[df1.columns], df[df2.columns])
        
        fname = f"datasets/ds_size{size}_feat{nf}.csv"
        df.to_csv(fname, index=False)
        print(f"{fname} | Shape: {df.shape} | Collision rate: {df['collision'].mean():.2%}")

datasets/ds_size50_feat5.csv | Shape: (50, 6) | Collision rate: 12.00%
datasets/ds_size50_feat9.csv | Shape: (50, 10) | Collision rate: 68.00%
datasets/ds_size50_feat12.csv | Shape: (50, 13) | Collision rate: 82.00%
datasets/ds_size300_feat5.csv | Shape: (300, 6) | Collision rate: 4.00%
datasets/ds_size300_feat9.csv | Shape: (300, 10) | Collision rate: 63.33%
datasets/ds_size300_feat12.csv | Shape: (300, 13) | Collision rate: 74.00%
datasets/ds_size750_feat5.csv | Shape: (750, 6) | Collision rate: 6.27%
datasets/ds_size750_feat9.csv | Shape: (750, 10) | Collision rate: 58.53%
datasets/ds_size750_feat12.csv | Shape: (750, 13) | Collision rate: 72.40%
datasets/ds_size1500_feat5.csv | Shape: (1500, 6) | Collision rate: 6.00%
datasets/ds_size1500_feat9.csv | Shape: (1500, 10) | Collision rate: 57.80%
datasets/ds_size1500_feat12.csv | Shape: (1500, 13) | Collision rate: 71.60%


In [28]:
def get_feature_groups(columns):
    binary = [c for c in columns if c.endswith('_binary')]
    nominal = [c for c in columns if c.endswith('_nominal')]
    ordinal = [c for c in columns if c.endswith('_ordinal')]
    quantitative = [c for c in columns if c.endswith('_quantitative')]
    return binary, nominal, ordinal, quantitative


def make_pipeline(model, feature_columns):
    binary, nominal, ordinal, quantitative = get_feature_groups(feature_columns)
    transformers = []
    if quantitative:
        transformers.append(('num', StandardScaler(), quantitative))
    if nominal:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal))
    preprocessor = ColumnTransformer(transformers=transformers, remainder='passthrough')
    return Pipeline([('preprocessor', preprocessor), ('classifier', model)])


def evaluate(pipe, X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - t0
    t1 = time.time()
    y_pred = pipe.predict(X_test)
    pred_time = time.time() - t1
    return {
        'f1': f1_score(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred),
        'fit_time': fit_time,
        'predict_time': pred_time
    }

Обоснование выбора 4 классических методов:
1. LogisticRegression: быстрая линейная модель, интерпретируема, служит надежным бенчмарком.
2. DecisionTreeClassifier: не требует масштабирования, устойчив к нелинейным зависимостям.
3. KNeighborsClassifier: интуитивно подходит для задачи коллизий, так как близкие состояния в пространстве признаков часто ведут к схожим исходам.
4. RandomForestClassifier: ансамблевый метод, устойчив к переобучению, стабильно работает на выборках разного объема.

Выбранная метрика: F1-score, так как классы могут быть несбалансированы (частота коллизий от 4% до 82%), и нам важен баланс между precision и recall.

In [29]:
models = {
    'logreg': LogisticRegression(max_iter=1000, random_state=42),
    'dtree': DecisionTreeClassifier(random_state=42),
    'knn': KNeighborsClassifier(),
    'rf': RandomForestClassifier(n_estimators=50, random_state=42)
}

In [30]:
all_results = []
for size in sizes:
    for nf in n_features_list:
        path = f"datasets/ds_size{size}_feat{nf}.csv"
        df = pd.read_csv(path)
        X = df.drop('collision', axis=1)
        y = df['collision']
        cols = X.columns.tolist()
        
        for name, model in models.items():
            pipe = make_pipeline(model, cols)
            metrics = evaluate(pipe, X, y)
            all_results.append({'size': size, 'n_features': nf, 'model': name, **metrics})

results_df = pd.DataFrame(all_results)
results_sorted = results_df.sort_values(by=['model', 'size', 'n_features'])

results_sorted

c:\Users\danii\Desktop\git\master\practice\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\danii\Desktop\git\master\practice\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,size,n_features,model,f1,accuracy,fit_time,predict_time
1,50,5,dtree,0.0000,1.0000,0.0050,0.0025
5,50,9,dtree,0.9091,0.8667,0.0072,0.0041
9,50,12,dtree,0.8333,0.7333,0.0103,0.0054
13,300,5,dtree,0.5000,0.9778,0.0053,0.0030
17,300,9,dtree,0.7652,0.7000,0.0079,0.0040
21,300,12,dtree,0.7481,0.6333,0.0083,0.0043
25,750,5,dtree,1.0000,1.0000,0.0052,0.0025
29,750,9,dtree,0.7692,0.7200,0.0083,0.0040
33,750,12,dtree,0.8098,0.7244,0.0098,0.0044
37,1500,5,dtree,1.0000,1.0000,0.0050,0.0027


In [31]:
small_df = results_df[results_df['size'] <= 100]
summary_small = small_df.groupby('model').agg({'f1': 'mean', 'accuracy': 'mean', 'fit_time': 'mean', 'predict_time': 'mean'}).reset_index()
display(summary_small.round(4))
top3 = summary_small.nsmallest(3, 'fit_time')['model'].tolist()

summary = results_df.groupby('model').agg({'f1': 'mean', 'accuracy': 'mean', 'fit_time': 'mean', 'predict_time': 'mean'}).reset_index()
display(summary.round(4))
top3 = summary.nsmallest(3, 'fit_time')['model'].tolist()

print(f"Top-3 fastest on small  {top3}")

,model,f1,accuracy,fit_time,predict_time
0,dtree,0.5808,0.8667,0.0075,0.0040
1,knn,0.5572,0.8000,0.0065,0.0057
2,logreg,0.5396,0.8000,0.0117,0.0040
3,rf,0.5741,0.8222,0.0731,0.0083


,model,f1,accuracy,fit_time,predict_time
0,dtree,0.7355,0.8089,0.0080,0.0038
1,knn,0.6755,0.8215,0.0067,0.0075
2,logreg,0.5195,0.7680,0.0128,0.0039
3,rf,0.7966,0.8383,0.0785,0.0088


Top-3 fastest on small  ['knn', 'dtree', 'logreg']


In [32]:
param_grids = {
    'logreg': {'classifier__C': [0.1, 1, 10]},
    'dtree': {'classifier__max_depth': [3, 5, None]},
    'knn': {'classifier__n_neighbors': [3, 5, 7]},
}

In [33]:
tuning_results = []
for model_name in top3:
    for size in [50, 1500]:
        for nf in [5, 12]: 
            path = f"datasets/ds_size{size}_feat{nf}.csv"
            if not os.path.exists(path):
                continue
            df = pd.read_csv(path)
            X = df.drop('collision', axis=1)
            y = df['collision']
            cols = X.columns.tolist()
            base_model = models[model_name]
            pipe = make_pipeline(base_model, cols)
            grid = GridSearchCV(pipe, param_grids[model_name], cv=3, scoring='f1', n_jobs=-1)
            t0 = time.time()
            grid.fit(X, y)
            elapsed = time.time() - t0
            tuning_results.append({'model': model_name, 'size': size, 'n_features': nf, 'best_params': grid.best_params_, 'best_f1': grid.best_score_, 'tuning_time': elapsed})
            print(f"Tuning {model_name} | Size:{size} | Time:{elapsed:.2f}s | F1:{grid.best_score_:.3f}")

tuning_df = pd.DataFrame(tuning_results)
tuning_df

Tuning knn | Size:50 | Time:2.92s | F1:0.000
Tuning knn | Size:50 | Time:2.81s | F1:0.898
Tuning knn | Size:1500 | Time:1.91s | F1:0.875
Tuning knn | Size:1500 | Time:0.15s | F1:0.856
Tuning dtree | Size:50 | Time:0.05s | F1:0.433
Tuning dtree | Size:50 | Time:0.07s | F1:0.772
Tuning dtree | Size:1500 | Time:0.04s | F1:0.960
Tuning dtree | Size:1500 | Time:0.06s | F1:0.834
Tuning logreg | Size:50 | Time:0.05s | F1:0.333
Tuning logreg | Size:50 | Time:0.07s | F1:0.919
Tuning logreg | Size:1500 | Time:0.06s | F1:0.000
Tuning logreg | Size:1500 | Time:0.08s | F1:0.824


,model,size,n_features,best_params,best_f1,tuning_time
0,knn,50,5,{'classifier__n_neighbors': 3},0.0000,2.9160
1,knn,50,12,{'classifier__n_neighbors': 3},0.8979,2.8068
2,knn,1500,5,{'classifier__n_neighbors': 3},0.8746,1.9105
3,knn,1500,12,{'classifier__n_neighbors': 7},0.8563,0.1541
4,dtree,50,5,{'classifier__max_depth': 5},0.4333,0.0540
5,dtree,50,12,{'classifier__max_depth': 3},0.7724,0.0705
6,dtree,1500,5,{'classifier__max_depth': None},0.9600,0.0449
7,dtree,1500,12,{'classifier__max_depth': 5},0.8343,0.0647
8,logreg,50,5,{'classifier__C': 10},0.3333,0.0491
9,logreg,50,12,{'classifier__C': 10},0.9192,0.0667


In [34]:
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ GRIDSEARCHCV")

for model in tuning_df['model'].unique():
    model_data = tuning_df[tuning_df['model'] == model]
    print(f"\n🔹 {model.upper()}:")
    print(f"   Средний F1: {model_data['best_f1'].mean():.4f} ± {model_data['best_f1'].std():.4f}")
    print(f"   Лучший F1:  {model_data['best_f1'].max():.4f}")
    print(f"   Общее время: {model_data['tuning_time'].sum():.3f}s")
    print(f"   Лучший параметр: {model_data.loc[model_data['best_f1'].idxmax(), 'best_params']}")

ИТОГОВЫЕ РЕЗУЛЬТАТЫ GRIDSEARCHCV

🔹 KNN:
   Средний F1: 0.6572 ± 0.4385
   Лучший F1:  0.8979
   Общее время: 7.787s
   Лучший параметр: {'classifier__n_neighbors': 3}

🔹 DTREE:
   Средний F1: 0.7500 ± 0.2251
   Лучший F1:  0.9600
   Общее время: 0.234s
   Лучший параметр: {'classifier__max_depth': None}

🔹 LOGREG:
   Средний F1: 0.5191 ± 0.4309
   Лучший F1:  0.9192
   Общее время: 0.252s
   Лучший параметр: {'classifier__C': 10}


`DessisionTree` - лучший выбор, затем идет `LogisticRegression`

In [2]:
import glob
all_files = glob.glob("datasets/ds_size*.csv")
largest_file = max(all_files, key=os.path.getsize)
print(f"Loading largest dataset: {largest_file}")

best_model_name = top3[0]  # knn

# Найти лучшие параметры из tuning_df
best_params_row = tuning_df[(tuning_df['model'] == best_model_name) & (tuning_df['size'] == 1500)]
best_params = best_params_row['best_params'].values[0] if not best_params_row.empty else {}

best_data = pd.read_csv(largest_file)
X_best = best_data.drop('collision', axis=1)
y_best = best_data['collision']
cols_best = X_best.columns.tolist()

# Создать пайплайн и ПРИМЕНИТЬ параметры
final_pipe = make_pipeline(models[best_model_name], cols_best)
if best_params:
    final_pipe.set_params(**best_params)  # ← ВАЖНО!
    
final_pipe.fit(X_best, y_best)

os.makedirs('models', exist_ok=True)
joblib.dump(final_pipe, f"models/best_{best_model_name}_final.pkl")
print(f"✓ Model saved: models/best_{best_model_name}_final.pkl")
print(f"  Parameters: {best_params}")

NameError: name 'os' is not defined